# MNIST API — Fase 1: Pipeline offline e serialização do modelo

**Objetivo:** treinar o classificador MLP vencedor do mini-projeto dentro de um
`sklearn.pipeline.Pipeline` e serializá-lo com `joblib` para uso posterior pela API FastAPI.

**Por que um Pipeline e não só o classificador?**

O Pipeline empacota *pré-processamento + modelo* em um único objeto. Isso garante
que a transformação aplicada nos dados de treino é **exatamente a mesma** aplicada
em produção (API). Se a normalização ficasse "solta" no notebook, alguém poderia
esquecer de aplicá-la na API — ou aplicar de outro jeito — e o modelo erraria
silenciosamente (distribuição de entrada divergente).

## 1. Imports e caminhos

Usamos `pathlib.Path` em vez de strings para caminhos:
- Mais legível e menos propenso a erros de `\` vs `/` (Windows vs Linux)
- `Path` resolve caminhos relativos ao notebook de forma portável
- Se o notebook rodar de outra pasta, `Path("models")` ainda aponta para o lugar certo

In [36]:
import json
import platform
import subprocess
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import sklearn
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
    roc_auc_score,
)

# Importa a função de normalização do módulo compartilhado
# (esse import funciona porque app/ é um pacote — veja app/__init__.py)
from app.preprocessing import normalizar_pixels

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

print(f"scikit-learn: {sklearn.__version__}")
print(f"Python: {platform.python_version()}")

scikit-learn: 1.9.1
Python: 3.13.15


### Por que `app/preprocessing.py` e não uma função definida aqui no notebook?

Quando fazemos `joblib.dump(pipeline, "modelo.joblib")`, o joblib serializa o
*objeto* Pipeline — incluindo referências aos métodos/funções que ele chama.
Para `FunctionTransformer`, ele grava o caminho importável da função
(ex.: `app.preprocessing.normalizar_pixels`).

Ao carregar com `joblib.load()` em **outro processo** (a API FastAPI), o Python
precisa importar essa função de volta. Se ela estivesse definida só neste notebook
(como variável local), o import falharia:

```
AttributeError: Can't get attribute 'normalizar_pixels' on <module '__main__'>
```

Criar `app/__init__.py` (mesmo vazio) transforma `app/` em um **pacote Python**,
tornando o import estável em qualquer processo que tenha o diretório no `sys.path`.

## 2. Carregamento dos dados e divisão estratificada 70/10/20

**Divisão estratificada** mantém a proporção de cada dígito (0–9) em cada subconjunto.
Sem isso, um dígito raro poderia acabar todo no teste e inflar/derrubar as métricas
de forma não representativa.

**Isso é leakage?** Não. A normalização `X/255.0` é uma transformação **fixa** —
ela não "aprende" nada dos dados (diferente de `StandardScaler`, que calcula média/desvio
do conjunto de treino). Aplicar `X/255` antes do split, dentro do Pipeline ou depois
não introduz viés. A única razão de colocá-la *dentro* do Pipeline é **consistência**
com a API, não prevenção de leakage.

**Nota sobre o cache do MNIST:** `fetch_openml` baixa o dataset (dezenas de MB)
e o guarda no cache do usuário (`~/scikit_learn_data`). Isso evita versionar
arquivos grandes no repositório, mas exige internet na primeira execução.

In [37]:
# MNIST via OpenML (mesmo dataset do scikit-learn, sem depender do fetch_mldata antigo)
mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")
X, y = mnist.data, mnist.target.astype("int64")

print(f"Shape de X: {X.shape}")
print(f"Shape de y: {y.shape}")
print(f"Classes: {np.unique(y)}")

Shape de X: (70000, 784)
Shape de y: (70000,)
Classes: [0 1 2 3 4 5 6 7 8 9]


In [38]:
# Divisão estratificada em 3 partes: treino (70%), validação (10%), teste (20%)
#
# Estratégia: dividimos 2 vezes de forma estratificada
#   1º split: 80% (treino+val) vs 20% (teste)
#   2º split: dos 80%, separamos 12.5% como validação → 70% total treino, 10% total validação

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.125, random_state=42, stratify=y_temp
)  # 0.125 de 80% = 10% do total

print(f"Treino:    {X_train.shape[0]:>6} amostras")
print(f"Validação: {X_val.shape[0]:>6} amostras")
print(f"Teste:     {X_test.shape[0]:>6} amostras")

Treino:     49000 amostras
Validação:   7000 amostras
Teste:      14000 amostras


## 3. Construção do Pipeline (normalização + MLP)

**Hiperparâmetros do MLP vencedor** (escolhidos no mini-projeto com base na validação;
aqui apenas reproduzimos):
- `hidden_layer_sizes=(128,)` — uma camada oculta com 128 neurônios
- `activation="relu"` — não-linearidade padrão, evita vanishing gradient
- `solver="adam"` — otimizador adaptativo, bom para grandes volumes
- `learning_rate_init=0.001` — taxa de aprendizado inicial
- `alpha=0.0001` — regularização L2 (weight decay) para reduzir overfitting
- `max_iter=30` — épocas máximas
- `early_stopping=True` — reserva 10% do próprio X_train como validação interna
  e interrompe o treino se a acurácia de validação interna não melhorar
  por `n_iter_no_change=10` épocas consecutivas
- `random_state=42` — reprodutibilidade

**Sobre `early_stopping=True`:** o MLP separa `validation_fraction=0.1` (10%) do
**próprio X_train** como validação interna. Ele **não** usa o nosso `X_val` externo.
Nosso `X_val` serve para avaliação pós-treino (comparar com o teste e checar overfitting).

**Por que `FunctionTransformer`?**
O Pipeline só aceita *estimadores* (com `.fit`/`.transform` ou `.predict`).
`FunctionTransformer` embrulha uma função simples como um transformador compatível,
permitindo incluir nossa normalização fixa no mesmo objeto do modelo.

In [39]:
pipeline = Pipeline(
    steps=[
        ("normalizacao", FunctionTransformer(normalizar_pixels, validate=False)),
        (
            "mlp",
            MLPClassifier(
                hidden_layer_sizes=(128,),
                activation="relu",
                solver="adam",
                learning_rate_init=0.001,
                alpha=0.0001,
                max_iter=30,
                early_stopping=True,
                random_state=42,
            ),
        ),
    ]
)

print(pipeline)

Pipeline(steps=[('normalizacao',
                 FunctionTransformer(func=<function normalizar_pixels at 0x000001F3217AF6A0>)),
                ('mlp',
                 MLPClassifier(early_stopping=True, hidden_layer_sizes=(128,),
                               max_iter=30, random_state=42))])


In [40]:
# Treino: o Pipeline aplica a normalização automaticamente em X_train.
# O early_stopping interno do MLP usa 10% do próprio X_train (validation_fraction),
# NÃO o nosso X_val externo.
#
# Suprimimos apenas o ConvergenceWarning (max_iter=30 atingido):
# é esperado e documentado na saída abaixo.
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
    pipeline.fit(X_train, y_train)

mlp = pipeline.named_steps["mlp"]
n_iter = mlp.n_iter_
print(f"Pipeline treinado com sucesso.")
print(f"Épocas executadas: {n_iter} / {mlp.max_iter}")
if n_iter >= mlp.max_iter:
    print("Aviso: o limite de épocas foi atingido (ConvergenceWarning suprimido abaixo).")
    print("O early_stopping não interrompeu antes, mas a acurácia permanece estável.")
    print("Mantivemos max_iter=30 idêntico ao mini-projeto para reprodutibilidade.")

Pipeline treinado com sucesso.
Épocas executadas: 30 / 30
Aviso: o limite de épocas foi atingido (ConvergenceWarning suprimido abaixo).
O early_stopping não interrompeu antes, mas a acurácia permanece estável.
Mantivemos max_iter=30 idêntico ao mini-projeto para reprodutibilidade.


## 4. Avaliação nos conjuntos de treino, validação e teste

Reportamos a acurácia nos três conjuntos:
- **Treino** — o modelo viu essas amostras durante o fit; serve como referência.
- **Validação** — nunca vista no fit; comparada com o treino para checar overfitting
  e com o teste para checar consistência.
- **Teste** — nunca visto; avaliação final.

| Métrica | O que mede | Como interpretar |
|---|---|---|
| **Acurácia** | Proporção de acertos globais | % de dígitos classificados corretamente |
| **Precision (weighted)** | Dos que o modelo disse ser classe X, quantos realmente eram | Média ponderada por suporte; penaliza falsos positivos |
| **Recall (weighted)** | Dos que realmente são classe X, quantos o modelo acertou | Média ponderada; penaliza falsos negativos |
| **F1 (weighted)** | Média harmônica de precision e recall | Balanceia FP e FN; bom resumo único |
| **ROC-AUC (OVR)** | Capacidade de ranquear cada classe vs as demais | 1.0 = perfeito; ~0.5 = aleatório; média "one-vs-rest" |

**Por que weighted?** Cada dígito tem quantidade similar de amostras no MNIST, mas
"weighted" garante que classes com mais amostras não dominem a métrica se houvesse desbalanceamento.

In [41]:
# Avaliação em treino (referência), validação (consistência) e teste (avaliação final)
acc_train = pipeline.score(X_train, y_train)
acc_val = pipeline.score(X_val, y_val)

y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)

# Acurácia
acc = accuracy_score(y_test, y_pred)

# Precision / Recall / F1 (weighted)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average="weighted"
)

# ROC-AUC multiclasse (One-vs-Rest)
roc_auc = roc_auc_score(
    y_test, y_proba, multi_class="ovr", average="weighted"
)

print(f"Acurácia (treino):    {acc_train:.4f}")
print(f"Acurácia (validação): {acc_val:.4f}")
print(f"Acurácia (teste):     {acc:.4f}")
print(f"Diferença val-teste:  {abs(acc_val - acc):.4f}")
print(f"---")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1:        {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

Acurácia (treino):    0.9974
Acurácia (validação): 0.9730
Acurácia (teste):     0.9754
Diferença val-teste:  0.0024
---
Precision: 0.9755
Recall:    0.9754
F1:        0.9754
ROC-AUC:   0.9994


### Interpretação: overfitting e consistência

O gap entre treino (0.9974) e validação (0.9730) é de cerca de 2,4 pontos
percentuais — um overfitting leve, esperado para um MLP no MNIST com 49k amostras
de treino. Já a diferença entre validação (0.9730) e teste (0.9754) é de apenas
0.0024, indicando que os dois conjuntos são consistentes entre si.

Para reduzir o gap treino × validação, opções incluem aumentar `alpha` (regularização
L2), usar uma CNN (aproveita a estrutura espacial da imagem) ou aplicar data
augmentation (rotações, deslocamentos). Mantivemos os hiperparâmetros idênticos
ao mini-projeto para reprodutibilidade.

In [42]:
# Relatório detalhado por classe
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      1.00      0.98      1381
           1       0.98      0.99      0.98      1575
           2       0.98      0.97      0.98      1398
           3       0.98      0.97      0.97      1428
           4       0.98      0.97      0.97      1365
           5       0.97      0.97      0.97      1263
           6       0.99      0.98      0.98      1375
           7       0.97      0.97      0.97      1459
           8       0.98      0.95      0.97      1365
           9       0.96      0.98      0.97      1391

    accuracy                           0.98     14000
   macro avg       0.98      0.98      0.98     14000
weighted avg       0.98      0.98      0.98     14000



### Salvando as métricas em `models/metrics.json`

Incluímos:
- Timestamp (UTC) para rastreabilidade
- Versão do scikit-learn — o `.joblib` é sensível a isso
- Hiperparâmetros **lidos do próprio modelo** via `get_params()` — evita divergência
  entre o que está no JSON e o que foi realmente treinado
- `n_iter_` — quantas épocas o early stopping executou

In [43]:
metrics = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "sklearn_version": sklearn.__version__,
    "python_version": platform.python_version(),
    "dataset": "mnist_784",
    "split": {"train": 0.70, "val": 0.10, "test": 0.20, "random_state": 42},
    "model": "MLPClassifier",
    "hyperparameters": {
        "hidden_layer_sizes": list(mlp.hidden_layer_sizes),
        "activation": mlp.activation,
        "solver": mlp.solver,
        "learning_rate_init": mlp.learning_rate_init,
        "alpha": mlp.alpha,
        "max_iter": mlp.max_iter,
        "early_stopping": mlp.early_stopping,
        "random_state": mlp.random_state,
    },
    "n_epochs_executed": int(n_iter),
    "train_metrics": {
        "accuracy": round(acc_train, 4),
    },
    "validation_metrics": {
        "accuracy": round(acc_val, 4),
    },
    "test_metrics": {
        "accuracy": round(acc, 4),
        "precision_weighted": round(precision, 4),
        "recall_weighted": round(recall, 4),
        "f1_weighted": round(f1, 4),
        "roc_auc_ovr_weighted": round(roc_auc, 4),
    },
}

metrics_path = MODELS_DIR / "metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(f"Métricas salvas em {metrics_path}")
print(json.dumps(metrics, indent=2))

Métricas salvas em models\metrics.json
{
  "timestamp_utc": "2026-10-17T20:02:24.612847+00:00",
  "sklearn_version": "1.9.1",
  "python_version": "3.13.15",
  "dataset": "mnist_784",
  "split": {
    "train": 0.7,
    "val": 0.1,
    "test": 0.2,
    "random_state": 42
  },
  "model": "MLPClassifier",
  "hyperparameters": {
    "hidden_layer_sizes": [
      128
    ],
    "activation": "relu",
    "solver": "adam",
    "learning_rate_init": 0.001,
    "alpha": 0.0001,
    "max_iter": 30,
    "early_stopping": true,
    "random_state": 42
  },
  "n_epochs_executed": 30,
  "train_metrics": {
    "accuracy": 0.9974
  },
  "validation_metrics": {
    "accuracy": 0.973
  },
  "test_metrics": {
    "accuracy": 0.9754,
    "precision_weighted": 0.9755,
    "recall_weighted": 0.9754,
    "f1_weighted": 0.9754,
    "roc_auc_ovr_weighted": 0.9994
  }
}


## 5. Serialização com joblib

O `joblib` é preferível ao `pickle` para objetos do scikit-learn porque:
- Lida melhor com arrays numpy grandes (os pesos da rede neural são arrays)
- É o formato recomendado pela própria documentação do sklearn

**Atenção:** o `.joblib` grava referências ao código (caminhos de import), não o
código em si. Por isso a dependência de `app/preprocessing.py` existir em qualquer
processo que carregue o modelo.

In [44]:
model_path = MODELS_DIR / "mnist_mlp_pipeline.joblib"
joblib.dump(pipeline, model_path)
print(f"Pipeline salvo em {model_path} ({model_path.stat().st_size / 1e6:.2f} MB)")

Pipeline salvo em models\mnist_mlp_pipeline.joblib (2.45 MB)


### Validação da serialização

Para testar de forma honesta que o modelo carrega em **outro processo** (como a API
fará), executamos um subprocess Python limpo via `subprocess`. Diferente de simplesmente
fazer `joblib.load()` neste mesmo kernel (onde `app.preprocessing` já foi importado),
o subprocess começa do zero — se a importação falhar, veremos o erro.

In [45]:
# Teste em processo limpo (subprocess) — simula o que a API fará
resultado = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import joblib; "
            f"p = joblib.load('{model_path.as_posix()}'); "
            "print(type(p).__name__)"
        ),
    ],
    capture_output=True,
    text=True,
    cwd=Path.cwd(),  # garante que app/ está no sys.path do subprocess
)

if resultado.returncode == 0:
    print(f"Subprocesso OK — modelo carregado: {resultado.stdout.strip()}")
else:
    print(f"ERRO no subprocesso:\n{resultado.stderr}")

Subprocesso OK — modelo carregado: Pipeline


In [46]:
# Além do subprocesso, validamos a consistência das previsões:
# carregamos do disco e comparamos com o pipeline em memória.
pipeline_carregado = joblib.load(model_path)

# Seleciona 5 amostras de teste (seed fixa para reprodutibilidade)
idx = np.random.RandomState(0).choice(len(X_test), size=5, replace=False)
X_amostras = X_test[idx]
y_amostras_real = y_test[idx]

pred_memoria = pipeline.predict(X_amostras)
pred_carregado = pipeline_carregado.predict(X_amostras)

print("Índice | Real | Em memória | Carregado")
print("-" * 45)
for i, real, p_mem, p_load in zip(idx, y_amostras_real, pred_memoria, pred_carregado):
    flag = "OK" if p_mem == p_load else "DIVERGIU!"
    print(f"{i:>6} | {real:>4} | {p_mem:>10} | {p_load:>9}  [{flag}]")

assert np.array_equal(pred_memoria, pred_carregado), \
    "As previsões divergiram! A serialização falhou."
print("\nSerialização validada: previsões idênticas.")

Índice | Real | Em memória | Carregado
---------------------------------------------
  6038 |    3 |          3 |         3  [OK]
 10758 |    8 |          8 |         8  [OK]
 13882 |    1 |          1 |         1  [OK]
  2760 |    5 |          5 |         5  [OK]
 10778 |    4 |          4 |         4  [OK]

Serialização validada: previsões idênticas.


## 6. Cuidados de reprodutibilidade

1. **Caminhos relativos com `pathlib`** — já feito acima; funciona em qualquer SO.
2. **`random_state=42`** em tudo que é estocástico (split + MLP) — mesma entrada, mesmo resultado.
3. **Versões no `requirements.txt`** — no fim do projeto, após instalar todas as
   dependências da API, rode no terminal (a partir da raiz do repositório):
   ```bash
   pip freeze > requirements.txt
   ```
   Isso congela todas as versões instaladas (inclusive `scikit-learn==`), evitando
   "funciona na minha máquina".
4. **Sensibilidade do joblib à versão do scikit-learn** — um `.joblib` criado com
   sklearn 1.9.0 pode falhar ao carregar em 1.9.1. Por isso salvamos a versão em
   `metrics.json` e fixaremos no `requirements.txt`.

### Pré-requisito

Este notebook deve ser executado com a **raiz do repositório** como diretório de
trabalho, pois o `from app.preprocessing import ...` depende disso.

### Checklist da Fase 1
- [x] Pipeline com normalização interna (`FunctionTransformer` + `app.preprocessing`)
- [x] Divisão estratificada 70/10/20
- [x] MLP treinado com hiperparâmetros do vencedor
- [x] Métricas na validação e no teste (acurácia, P/R/F1 weighted, ROC-AUC OVR)
- [x] `models/metrics.json` salvo (com `n_iter_` e hiperparâmetros do modelo)
- [x] `models/mnist_mlp_pipeline.joblib` salvo e validado (subprocesso limpo + consistência)
- [x] Estrutura de pastas `app/`, `models/`, `data/meus_digitos/` criada

In [47]:
# Resumo final
print("=== Fase 1 concluída ===")
print(f"Modelo:  {model_path}")
print(f"Métricas: {metrics_path}")
print(f"Épocas executadas: {n_iter}")
print(f"Acurácia (treino):    {acc_train:.4f}")
print(f"Acurácia (validação): {acc_val:.4f}")
print(f"Acurácia (teste):     {acc:.4f}")
print(f"F1 weighted:          {f1:.4f}")
print(f"ROC-AUC OVR:          {roc_auc:.4f}")

=== Fase 1 concluída ===
Modelo:  models\mnist_mlp_pipeline.joblib
Métricas: models\metrics.json
Épocas executadas: 30
Acurácia (treino):    0.9974
Acurácia (validação): 0.9730
Acurácia (teste):     0.9754
F1 weighted:          0.9754
ROC-AUC OVR:          0.9994
